In [ ]:
# the solving of apollo code is already proven we dont have to do that
# but our value add is to see who is generating the code + specification better
# we will compare what is the required depth needed for each of the model
# see which task type is having difficulty proving it
# which model is better

# sicne apollo also requires gpu we will have to generte the lean 3 times first for different seed for each model
# save it as a df for each of the models the 9 models that you are trying to do it for
# use the same prompt template as what we used for claude code

# after it is done then you solve the apollo and get the result -> for each of the experiment 

In [6]:
import pandas as pd
a = pd.read_csv('dataset.csv')

In [7]:
a.head()

,property,expected_verdict,c_file,c_file_abs,yml_path,folder
0,def-behavior,False,c/termination-crafted/Arrays03-ValueRestictsIn...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays03-ValueRestictsIn...,termination-crafted
1,def-behavior,False,c/termination-crafted/Nyala-2lex-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Nyala-2lex-1.yml,termination-crafted
2,def-behavior,False,c/termination-crafted/LexIndexValue-Array-2.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/LexIndexValue-Array-2.yml,termination-crafted
3,def-behavior,False,c/termination-crafted/Arrays01-EquivalentConst...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays01-EquivalentConst...,termination-crafted
4,def-behavior,False,c/termination-crafted/SyntaxSupportPointer01-3.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/SyntaxSupportPointer01-3...,termination-crafted


In [20]:
import pandas as pd
import numpy as np

def get_property_prioritized_sample(df, total_n=100):
    properties = df['property'].unique()
    n_props = len(properties)
    
    # Calculate how many samples we want per property
    samples_per_prop = total_n // n_props
    samples_per_verdict = samples_per_prop // 2
    
    final_indices = []

    for prop in properties:
        prop_df = df[df['property'] == prop]
        
        for verdict in [True, False]:
            # Filter by property and verdict
            sub_df = prop_df[prop_df['expected_verdict'] == verdict]
            
            if sub_df.empty:
                continue
            
            # To maximize folder spread, we shuffle the folders
            # then take the first occurrence of each unique folder first
            sub_df = sub_df.sample(frac=1, random_state=42)
            
            # This prioritizes unique folders within this property/verdict bucket
            diverse_folders = sub_df.drop_duplicates(subset=['folder'])
            
            if len(diverse_folders) >= samples_per_verdict:
                picked = diverse_folders.head(samples_per_verdict)
            else:
                # If we don't have enough unique folders, take them all 
                # and fill the rest from the remaining rows
                remaining_needed = samples_per_verdict - len(diverse_folders)
                other_rows = sub_df.drop(diverse_folders.index)
                
                # Take as many as possible if the pool is smaller than needed
                n_to_take = min(len(other_rows), remaining_needed)
                fillers = other_rows.head(n_to_take)
                picked = pd.concat([diverse_folders, fillers])
            
            final_indices.extend(picked.index.tolist())

    # Create the sample dataframe
    sample_df = df.loc[final_indices].copy()
    
    # --- Emergency Top-up ---
    # If some properties didn't have enough True/False samples, 
    # we might be under 100. Let's fill the gap.
    if len(sample_df) < total_n:
        gap = total_n - len(sample_df)
        remaining_pool = df.drop(sample_df.index)
        # Try to maintain balance even in the top-up
        top_up = remaining_pool.sample(n=min(gap, len(remaining_pool)), random_state=42)
        sample_df = pd.concat([sample_df, top_up])

    return sample_df.sample(frac=1, random_state=42).head(total_n)

# Run it on your dataframe 'a'
balanced_sample = get_property_prioritized_sample(a, 50)

# Verify the results
print(f"Final Count: {len(balanced_sample)}")
print("\n--- Verdict Balance ---")
print(balanced_sample['expected_verdict'].value_counts())
print("\n--- Samples per Property ---")
print(balanced_sample['property'].value_counts())

Final Count: 50

--- Verdict Balance ---
expected_verdict
False    27
True     23
Name: count, dtype: int64

--- Samples per Property ---
property
no-overflow        12
termination        12
unreach-call       11
valid-memsafety    10
def-behavior        5
Name: count, dtype: int64


In [21]:
balanced_sample.to_csv('c_sample.csv')

In [19]:
pd.read_csv('./dataset_temp_02.csv').iloc[0].lean_proof_seed_999

'import Mathlib\n\nopen Lean.Elab.Tactics\nopen Lean.Parser.Tactics\nopen Lean.Parser.Term\n\ndef init (main : Nat) : Unit := do\n  let N := main\n  assume N > 0\n  assume N <= 2147483647 / sizeOf <int>\n  let a : Array Int64 := Array.fill! (Array.ofSize N Int64.zero)\n  for i in 0 .. N - 1 do\n    a.update! i (i - 1) * (i + 1)\n  for i in 0 .. N - 1 do\n    a.update! i (a.get! i - i * i)\n  for i in 0 .. N - 1 do\n    assume a.get! i = 0\n\ndef check : Unit => Tactic := do\n  let ℵ0 := ∀ (n : Nat), n ≠ 0\n  let P := ∀ (i : Nat), a.get! i = 0\n  let Q := ∀ (i : Nat), i < N → ∀ (j : Nat), i ≠ j → a.get! i ≠ a.get! j\n  let R := ∀ (i : Nat), i < N → a.get! i = i - 1\n  let G := ∀ (s : StateM ℕ), ℵ0 s.memory.N → P s.memory.a\n  let F := ∀ (s : StateM ℕ), ℵ0 s.memory.N → Q s.memory.a s.memory.N\n  let H := ∀ (s : StateM ℕ), ℵ0 s.memory.N → R s.memory.a s.memory.N\n  let I := ∀ (s : StateM ℕ), ℵ0 s.memory.N → G s.memory.a → F s.memory.a → H s.memory.a → pure unit\n  let C := ∀ (s : StateM ℕ

In [1]:
import os
import sys
os.chdir('/workspace/APOLLO')
sys.path.insert(0, '/workspace/APOLLO')   # for apollo imports
sys.path.insert(0, '/workspace/CS527-Project/')   # for your cs527 scripts

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path

def authenticate_hf():
    
    # 2. Define the path relative to the script
    # If .env is one folder up:
    env_path = "/workspace/.env"
    load_dotenv(dotenv_path=env_path)
    
    # Get the token from environment variables
    hf_token = os.getenv("HF_TOKEN")
    
    if hf_token:
        # Login to Hugging Face
        # add 'add_to_git_credential=True' if you want to save the token to your machine
        login(token=hf_token)
        print("Successfully logged into Hugging Face.")
    else:
        print("Error: HF_TOKEN not found in .env file.")

In [4]:
authenticate_hf()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged into Hugging Face.


In [5]:
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Meta-Llama-3-8B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': "I am LLaMA, an AI assistant developed by Meta AI that can understand and respond to human input in a conversational manner. I'm not a human, but a computer program designed to simulate conversation, answer questions, and even generate text based on the input I receive.\n\nI'm a type of language model, trained on a massive dataset of text from the internet, books, and other sources. This training enables me to understand and respond to a wide range of topics, from science and history to entertainment and culture.\n\nMy primary function is to assist and provide information to users through text-based conversations. I can help with tasks such as:\n\n1. Answering questions on various topics\n2. Generating text based on a prompt or topic\n3. Translation between languages\n4. Summarizing long pieces of text\n5. Offering suggestions and recommendations\n\nI'm constantly learning and impr

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('/workspace/CS527-Project/dataset.csv')
df.head()

,property,expected_verdict,c_file,c_file_abs,yml_path,folder
0,def-behavior,False,c/termination-crafted/Arrays03-ValueRestictsIn...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays03-ValueRestictsIn...,termination-crafted
1,def-behavior,False,c/termination-crafted/Nyala-2lex-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Nyala-2lex-1.yml,termination-crafted
2,def-behavior,False,c/termination-crafted/LexIndexValue-Array-2.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/LexIndexValue-Array-2.yml,termination-crafted
3,def-behavior,False,c/termination-crafted/Arrays01-EquivalentConst...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays01-EquivalentConst...,termination-crafted
4,def-behavior,False,c/termination-crafted/SyntaxSupportPointer01-3.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/SyntaxSupportPointer01-3...,termination-crafted


In [7]:
# read in the c file as well as the yml file for the specificaiton

with open(df.c_file_abs.loc[5], 'r', encoding='utf-8') as file:
        c_code_string = file.read()

# read in the yml_file
property_path = f'/workspace/sv-benchmarks/c/properties/{df.property.loc[5]}.prp'
with open(property_path, 'r', encoding='utf-8') as file:
        property_string = file.read()

In [9]:

from build_prompt import build_prompt

In [10]:
system_prompt, user_prompt = build_prompt(
    c_code=c_code_string,
    specification=property_string,       # contents of the .prp file
)

In [11]:
# 2. Structure them as a list of messages
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# 3. Pass to the chatbot
# Note: set 'return_full_text=False' to get only the new response
output = pipe(
    messages, 
    max_new_tokens=1024, 
    do_sample=True, 
    temperature=0.2,
    return_full_text=False
)

# 4. Access the content
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


```
import Mathlib
import Aesop

set_option maxHeartbeats 0

open BigOperators Real Nat Topology Rat

inductive bool : Type
| ff : bool
| tt : bool

def nondet_int : Type := ∀ (x : Int), Prop

def main_lean : nondet_int → nondet_int → nondet_int → Int → Int → Int → Prop
| _ _ _ tx x y := 
  (tx ≤ 1073741823) ∧ (x ≤ 1073741823) ∧ (y ≥ -1073741823) ∧
  (∀ (i : ℕ), i < (x - y) → tx ≥ i) ∧
  (∀ (i : ℕ), i < (y - x) → tx ≤ i)

def main_spec : Prop := 
  ∀ (tx x y : Int), main_lean tx x y → 
  (∀ (i : ℕ), i < (x - y) → tx ≥ i) ∧ 
  (∀ (i : ℕ), i < (y - x) → tx ≤ i) → False
```


In [7]:
import torch
import gc

# Delete your pipeline
# del pipe        # or whatever your model variable is called
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("GPU freed:", torch.cuda.memory_allocated() / 1e6, "MB used")

GPU freed: 0.0 MB used


In [8]:
# use apollo repair
import os
from apollo import ApolloRepair


# code  ='''
# import Mathlib
# import Aesop

# set_option maxHeartbeats 0

# open BigOperators Real Nat Topology Rat

# theorem mathd_algebra_141 (a b : ℝ) (h₁ : a * b = 180) (h₂ : 2 * (a + b) = 54) :
#     a ^ 2 + b ^ 2 = 369 := by
#   have h3 : a + b = 27 := by
#     field_simp [h₂]
#   have h4 : (a + b) ^ 2 = 729 := by
#     rw [h3]
#     norm_num
#   have expand : a ^ 2 + b ^ 2 = (a + b) ^ 2 - 2 * a * b := by
#     ring
#   have step1 : a ^ 2 + b ^ 2 = 729 - 2 * a * b := by
#     rw [expand, h4]
#   have step2 : 729 - 2 * a * b = 729 - 360 := by
#     rw [h₁]
#   have step3 : 729 - 360 = 369 := by
#     norm_num
#   exact step1.trans (step2.trans step3)
# '''

code = """
import Mathlib

-- The C int range: [-2^31, 2^31-1]
def INT_MIN : Int := -2147483648
def INT_MAX : Int := 2147483647

-- The initial bounds guaranteed by the C code's overflow-prevention checks
def BOUND : Int := 1073741823  -- 2^30 - 1

-- Loop invariant maintained throughout execution:
-- tx is non-negative and bounded above; x is bounded above; y is bounded below
def loopInv (x y tx : Int) : Prop :=
  0 ≤ tx ∧ tx ≤ BOUND ∧ x ≤ BOUND ∧ -BOUND ≤ y

-- A value is in the valid C int range (no overflow)
def inRange (v : Int) : Prop :=
  INT_MIN ≤ v ∧ v ≤ INT_MAX

-- The x-branch update (x = x - 1 - tx) stays in range
theorem x_update_no_overflow (x y tx : Int)
    (hInv : loopInv x y tx) (hCond : x ≥ y) :
    inRange (x - 1 - tx) := by
  unfold loopInv at hInv
  unfold inRange INT_MIN INT_MAX BOUND at *
  obtain ⟨h1, h2, h3, h4⟩ := hInv
  constructor <;> linarith

-- The y-branch update (y = y + 1 + tx) stays in range
theorem y_update_no_overflow (x y tx : Int)
    (hInv : loopInv x y tx) (hCond : x ≥ y) :
    inRange (y + 1 + tx) := by
  unfold loopInv at hInv
  unfold inRange INT_MIN INT_MAX BOUND at *
  obtain ⟨h1, h2, h3, h4⟩ := hInv
  constructor <;> linarith

-- The x-branch update preserves the loop invariant
theorem x_update_preserves_inv (x y tx : Int)
    (hInv : loopInv x y tx) (hCond : x ≥ y) :
    loopInv (x - 1 - tx) y tx := by
  unfold loopInv at *
  unfold BOUND at *
  obtain ⟨h1, h2, h3, h4⟩ := hInv
  exact ⟨h1, h2, by linarith, h4⟩

-- The y-branch update preserves the loop invariant
theorem y_update_preserves_inv (x y tx : Int)
    (hInv : loopInv x y tx) (hCond : x ≥ y) :
    loopInv x (y + 1 + tx) tx := by
  unfold loopInv at *
  unfold BOUND at *
  obtain ⟨h1, h2, h3, h4⟩ := hInv
  exact ⟨h1, h2, h3, by linarith⟩

-- The initial checks in the C program establish the loop invariant
theorem initial_conditions_establish_inv (tx x y : Int)
    (hTx : tx ≤ BOUND) (hX : x ≤ BOUND) (hY : -BOUND ≤ y) (hTxNn : 0 ≤ tx) :
    loopInv x y tx :=
  ⟨hTxNn, hTx, hX, hY⟩

-- Main safety theorem:
-- Given the initial preconditions from the C program's overflow checks,
-- no overflow can occur in any loop iteration (LTL: G ! overflow).
-- We model reachable states via the loop invariant: any state satisfying
-- loopInv is reachable (inductive over loop iterations).
theorem main_no_overflow
    (tx x y : Int)
    (hTx : tx ≤ BOUND) (hX : x ≤ BOUND) (hY : -BOUND ≤ y) (hTxNn : 0 ≤ tx) :
    -- For all reachable states satisfying the invariant:
    ∀ (x' y' : Int),
      loopInv x' y' tx →
      -- When the loop condition holds (x >= y && tx >= 0):
      x' ≥ y' →
      -- Both branches of the loop body are overflow-free:
      inRange (x' - 1 - tx) ∧ inRange (y' + 1 + tx) :=
  fun x' y' hInv hCond =>
    ⟨x_update_no_overflow x' y' tx hInv hCond,
     y_update_no_overflow x' y' tx hInv hCond⟩

-- Corollary: the invariant is inductive — it holds at every loop iteration
theorem inv_is_inductive (tx : Int) :
    ∀ (x y : Int),
      loopInv x y tx →
      x ≥ y →
      -- After either branch, the invariant still holds
      loopInv (x - 1 - tx) y tx ∧ loopInv x (y + 1 + tx) tx :=
  fun x y hInv hCond =>
    ⟨x_update_preserves_inv x y tx hInv hCond,
     y_update_preserves_inv x y tx hInv hCond⟩
"""



# Parameters
max_attempts = 10 # maximum recursion depth
config = 'configs/baseline_sampling_kimina_prover.py' # config file for LLM
problem_dir = 'logs/test/idk' # where to save final lean file and intermediate proof states

# Instantiate the Apollo repair object
manager = ApolloRepair(
    code=code,
    lemma_name='test',
    config=config,
    rec_depth=max_attempts,
    log_dir = problem_dir
)

# Run the entire fix/verify pipeline and get the final assembled proof path
final_proof_path = manager.run()

with open(final_proof_path, 'r') as f:
    proof = f.read()
print(proof)

[test] Attempt 1, after syntax fix:
import Mathlib
def INT_MIN : Int := by -2147483648
def INT_MAX : Int := by 2147483647
def BOUND : Int := by 1073741823 -- 2^30 - 1
def loopInv (x y tx : Int) : Prop := by 
  0 ≤ tx ∧ tx ≤ BOUND ∧ x ≤ BOUND ∧ -BOUND ≤ y
def inRange (v : Int) : Prop := by 
  INT_MIN ≤ v ∧ v ≤ INT_MAX
set_option pp.instanceTypes true
set_option pp.numericTypes true
set_option pp.coercions.types true
set_option pp.letVarTypes true
set_option pp.structureInstanceTypes true
set_option pp.instanceTypes true
set_option pp.mvars.withType true
set_option pp.coercions true
set_option pp.funBinderTypes true
set_option pp.piBinderTypes true
theorem x_update_no_overflow (x y tx : Int)
    (hInv : loopInv x y tx) (hCond : x ≥ y) :
    inRange (x - 1 - tx) := by
  unfold loopInv at hInv
  unfold inRange INT_MIN INT_MAX BOUND at *
  obtain ⟨h1, h2, h3, h4⟩ := by hInv
  constructor <;> linarith
theorem y_update_no_overflow (x y tx : Int)
    (hInv : loopInv x y tx) (hCond : x ≥ y) :
 

Fixing with Sorrifier: 0 cycles [00:05]

Following Error occured in Sorrifier block: Problem statetement corrupted!
[test] After sorrification:
import Mathlib
def INT_MIN : Int := by
  sorry

Begin HintRepair...



Correcting with other solvers: 100%|██████████| 3/3 [00:00<00:00, 42224.54it/s]


[test] Final repaired code:
import Mathlib
def INT_MIN : Int := by
  exact Aesop.defaultSafePenalty



0it [00:00, ?it/s]

[test] Attempt 1 - extracted 0 lemmas.
[test] Attempt 1 - formatted into 0 JSON lines.


Number of TODO Problems: 0
Complete launching 8 LeanServerProcesses
Complete launching 0 SearchProcesses


Process GeneratorProcess-30:
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/workspace/APOLLO/prover/workers/generator.py", line 42, in run
    torch.cuda.set_device(f'cuda:0')  # Explicitly set the GPU for this process
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/torch/cuda/__init__.py", line 584, in set_device
    torch._C._cuda_setDevice(device)
  File "/venv/main/lib/python3.12/site-packages/torch/cuda/__init__.py", line 412, in _lazy_init
    raise RuntimeError(
RuntimeError: Cannot re-initialize CUDA in forked subprocess. To use CUDA with multiprocessing, you must use the 'spawn' start method


Complete launching 1 LLMProcesses
All 0 SearchProcesses stopped
All 8 LeanServerProcesses stopped
All 1 LLMProcesses stopped
['import Mathlib\n', 'def INT_MIN : Int := by\n', '  exact Aesop.defaultSafePenalty']
Final assembled proof saved to: logs/test/idk/assembled_main_theorem.lean
ApolloRepair finished in 56.70s.
import Mathlib
def INT_MIN : Int := by
  exact Aesop.defaultSafePenalty


In [10]:
from prover.lean.verifier import verify_lean4_file

In [14]:
verify_lean4_file(output[0]['generated_text'], timeout=30)

{'sorries': [],
 'tactics': [],
 'errors': [{'severity': 'error',
   'pos': {'line': 1, 'column': 0},
   'endPos': {'line': 1, 'column': 1},
   'data': "unexpected token '`'; expected command"},
  {'severity': 'error',
   'pos': {'line': 2, 'column': 0},
   'endPos': {'line': 2, 'column': 6},
   'data': "invalid 'import' command, it must be used in the beginning of the file"}],
 'warnings': [],
 'infos': [],
 'system_messages': '',
 'system_errors': None,
 'ast': {},
 'verified_code': '```\nimport Mathlib\nimport Aesop\n\nset_option maxHeartbeats 0\n\nopen BigOperators Real Nat Topology Rat\n\ninductive bool : Type\n| ff : bool\n| tt : bool\n\ndef nondet_int : Type := ∀ (x : Int), Prop\n\ndef main_lean : nondet_int → nondet_int → nondet_int → Int → Int → Int → Prop\n| _ _ _ tx x y := \n  (tx ≤ 1073741823) ∧ (x ≤ 1073741823) ∧ (y ≥ -1073741823) ∧\n  (∀ (i : ℕ), i < (x - y) → tx ≥ i) ∧\n  (∀ (i : ℕ), i < (y - x) → tx ≤ i)\n\ndef main_spec : Prop := \n  ∀ (tx x y : Int), main_lean tx x y 

In [11]:
verify_lean4_file(proof, timeout=30)

{'sorries': [],
 'tactics': [],
 'errors': [],
 'warnings': [],
 'infos': [],
 'system_messages': '',
 'system_errors': None,
 'ast': {},
 'verified_code': 'import Mathlib\ndef INT_MIN : Int := by\n  exact Aesop.defaultSafePenalty',
 'pass': True,
 'complete': True,
 'verify_time': 5.815457820892334}

In [14]:
output[0]['generated_text']

'```\nimport Mathlib\nimport Aesop\n\nset_option maxHeartbeats 0\n\nopen BigOperators Real Nat Topology Rat\n\ninductive BitVec (n : Nat) : Type\n| mk : Fin n → BitVec n\n\nnamespace BitVec\n\ndef toNat (n : Nat) (b : BitVec n) : Nat :=\nfin.val (b.mk)\n\nend BitVec\n\ndef main_lean : BitVec 32 → BitVec 32 → Prop\n| _ _ := True\n\ndef main_spec : ∀ (k : BitVec 32), main_lean k (mk 0 23) →\n  ∀ (x : BitVec 32), x ≥ 0 → x = 0 → main_lean k (mk k 42) → False\n:=\nbegin\n  intros k h1 h2 h3 h4,\n  unfold main_lean at h1 h4,\n  simp at h1 h4,\n  contradiction\nend\n```'

In [9]:
proof

'import Mathlib\ndef INT_MIN : Int := by\n  exact Aesop.defaultSafePenalty'

In [19]:
df

,property,expected_verdict,c_file,c_file_abs,yml_path,folder
0,def-behavior,False,c/termination-crafted/Arrays03-ValueRestictsIn...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays03-ValueRestictsIn...,termination-crafted
1,def-behavior,False,c/termination-crafted/Nyala-2lex-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Nyala-2lex-1.yml,termination-crafted
2,def-behavior,False,c/termination-crafted/LexIndexValue-Array-2.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/LexIndexValue-Array-2.yml,termination-crafted
3,def-behavior,False,c/termination-crafted/Arrays01-EquivalentConst...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Arrays01-EquivalentConst...,termination-crafted
4,def-behavior,False,c/termination-crafted/SyntaxSupportPointer01-3.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/SyntaxSupportPointer01-3...,termination-crafted
...,...,...,...,...,...,...
681,valid-memsafety,False,c/loops/insertion_sort-1.c,/workspace/sv-benchmarks/c/loops/insertion_sor...,c/loops/insertion_sort-1.yml,loops
682,valid-memsafety,False,c/array-memsafety/diff_usafe.c,/workspace/sv-benchmarks/c/array-memsafety/dif...,c/array-memsafety/diff_usafe.yml,array-memsafety
683,valid-memsafety,False,c/array-memsafety/selectionsort_unsafe.c,/workspace/sv-benchmarks/c/array-memsafety/sel...,c/array-memsafety/selectionsort_unsafe.yml,array-memsafety
684,valid-memsafety,False,c/loops/bubble_sort-1.c,/workspace/sv-benchmarks/c/loops/bubble_sort-1.c,c/loops/bubble_sort-1.yml,loops


In [20]:
df.query('expected_verdict==True')

,property,expected_verdict,c_file,c_file_abs,yml_path,folder
5,no-overflow,True,c/termination-crafted/aaron2-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/aaron2-1.yml,termination-crafted
6,no-overflow,True,c/termination-restricted-15/a.08.c,/workspace/sv-benchmarks/c/termination-restric...,c/termination-restricted-15/a.08.yml,termination-restricted-15
7,no-overflow,True,c/termination-crafted-lit/AliasDarteFeautrierG...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted-lit/AliasDarteFeautrierG...,termination-crafted-lit
8,no-overflow,True,c/termination-restricted-15/java_Sequence.c,/workspace/sv-benchmarks/c/termination-restric...,c/termination-restricted-15/java_Sequence.yml,termination-restricted-15
9,no-overflow,True,c/termination-nla/fermat2-both-nt.c,/workspace/sv-benchmarks/c/termination-nla/fer...,c/termination-nla/fermat2-both-nt.yml,termination-nla
...,...,...,...,...,...,...
665,valid-memsafety,True,c/termination-crafted/2Nested-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/2Nested-1.yml,termination-crafted
666,valid-memsafety,True,c/termination-crafted/RecursiveNonterminating-1.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/RecursiveNonterminating-...,termination-crafted
667,valid-memsafety,True,c/termination-crafted/Nyala-2lex-2.c,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Nyala-2lex-2.yml,termination-crafted
668,valid-memsafety,True,c/termination-crafted/Lobnya-Boolean-Reordered...,/workspace/sv-benchmarks/c/termination-crafted...,c/termination-crafted/Lobnya-Boolean-Reordered...,termination-crafted


In [23]:
df.loc[5]

property                                                  no-overflow
expected_verdict                                                 True
c_file                               c/termination-crafted/aaron2-1.c
c_file_abs          /workspace/sv-benchmarks/c/termination-crafted...
yml_path                           c/termination-crafted/aaron2-1.yml
folder                                            termination-crafted
Name: 5, dtype: object

In [19]:
prompt = f"""Translate this C code into a Lean 4 proof. Output Lean 4 code only.

## C Code
```c
{c_code_string}
```

## Specification
{property_string}

Begin your response with `import Mathlib` and nothing else before it.
Do not write any English text before or after the Lean 4 code block.
"""

messages = [
    {
        "role": "system",
        "content": "You output only Lean 4 code. No English. No explanations. No self-correction."
    },
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": "import Mathlib\n"}  # FORCE it to start here
]

In [20]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
sampling_params = SamplingParams(temperature=0.3, top_p=0.95, max_tokens=8096)
output = model.generate(text, sampling_params=sampling_params)
output_text = output[0].outputs[0].text
print(output_text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<think>
# Solving the C to Lean 4 Proof Problem

I'll tackle the problem of converting the given C code into a Lean 4 proof. Let's analyze the problem first.

We have a C program that uses `nondeterministic` integers (`__VERIFIER_nondet_int`) and we need to prove that the output of this program is always `0` given the constraints. The proof uses a `while` loop and `if` conditions with `nondeterministic` integers.

First, let's understand the structure of the C code:

```tactics
import Mathlib

```

We need to import Mathlib to access the necessary definitions and lemmas for working with integers, natural numbers, and basic arithmetic operations.

Now, let's look at the theorem statement:

```tactics
theorem main : (0 : Int) = (0 : Int) := by rfl
```

Wait, actually, looking at the problem description, it's `CHECK( init(main()), LTL(G ! overflow) )`. This suggests that the goal is to prove that the initial value of the main function is 0, and that the loop terminates without overflow.



In [12]:
df.loc[5]

property                                                  no-overflow
expected_verdict                                                 True
c_file                               c/termination-crafted/aaron2-1.c
c_file_abs          /workspace/sv-benchmarks/c/termination-crafted...
yml_path                           c/termination-crafted/aaron2-1.yml
folder                                            termination-crafted
Name: 5, dtype: object

In [16]:
c_code_string

'/*\n * Program used in the experimental evaluation of the following papers.\n * 2008ESOP - Chawdhary,Cook,Gulwani,Sagiv,Yang - Ranking Abstractions\n * 2010SAS - Alias,Darte,Feautrier,Gonnord, Multi-dimensional Rankings, Program \n *           Termination, and Complexity Bounds of Flowchart Programs\n *\n * Date: 2014\n * Author: Caterina Urban\n */\n\ntypedef enum {false, true} bool;\n\nextern int __VERIFIER_nondet_int(void);\n\nint main() {\n    int x, y, tx;\n\ttx = __VERIFIER_nondet_int();\n\tx = __VERIFIER_nondet_int();\n\ty = __VERIFIER_nondet_int();\n  //prevent overflows\n  if(!(tx<=1073741823)) return 0;\n  if(!(x<=1073741823)) return 0;\n  if(!(y>=-1073741823)) return 0;\n\twhile (x >= y && tx >= 0) {\n\t\tif (__VERIFIER_nondet_int() != 0) {\n\t\t\tx = x - 1 - tx;\n\t\t} else {\n\t\t\ty = y + 1 + tx;\n\t\t}\n\t}\n\treturn 0;\n}\n'

In [21]:
prompt

'Translate this C code into a Lean 4 proof. Output Lean 4 code only.\n\n## C Code\n```c\n/*\n * Program used in the experimental evaluation of the following papers.\n * 2008ESOP - Chawdhary,Cook,Gulwani,Sagiv,Yang - Ranking Abstractions\n * 2010SAS - Alias,Darte,Feautrier,Gonnord, Multi-dimensional Rankings, Program \n *           Termination, and Complexity Bounds of Flowchart Programs\n *\n * Date: 2014\n * Author: Caterina Urban\n */\n\ntypedef enum {false, true} bool;\n\nextern int __VERIFIER_nondet_int(void);\n\nint main() {\n    int x, y, tx;\n\ttx = __VERIFIER_nondet_int();\n\tx = __VERIFIER_nondet_int();\n\ty = __VERIFIER_nondet_int();\n  //prevent overflows\n  if(!(tx<=1073741823)) return 0;\n  if(!(x<=1073741823)) return 0;\n  if(!(y>=-1073741823)) return 0;\n\twhile (x >= y && tx >= 0) {\n\t\tif (__VERIFIER_nondet_int() != 0) {\n\t\t\tx = x - 1 - tx;\n\t\t} else {\n\t\t\ty = y + 1 + tx;\n\t\t}\n\t}\n\treturn 0;\n}\n\n```\n\n## Specification\nCHECK( init(main()), LTL(G ! over